In [4]:
import numpy as np

# -------------------------------------------------------------------
# Minimal dummy version of `icesee_get_index` for testing
# -------------------------------------------------------------------
def fake_icesee_get_index(**kwargs):
    statevec_true = kwargs["statevec_true"]
    vec_inputs = kwargs["vec_inputs"]
    ndim = statevec_true.shape[0] // len(vec_inputs)
    indx_map = {}

    for i, key in enumerate(vec_inputs):
        start = i * ndim
        end = start + ndim
        indx_map[key] = np.arange(start, end)

    return None, indx_map, None


# -------------------------------------------------------------------
# Minimal dummy observation-schedule generator
# -------------------------------------------------------------------
def fake_generate_observation_schedule(**kwargs):
    obs_times = kwargs["obs_times"]
    ind_m = obs_times
    m_obs = len(obs_times)
    return obs_times, ind_m, m_obs


# -------------------------------------------------------------------
# Test class with the synthetic observation generator
# -------------------------------------------------------------------
class TestICESEE:
    def __init__(self):
        self.params = {
            "total_state_param_vars": 5,
            "num_state_vars": 3,
            "sig_obs": [0.1, 0.1, 0.1],
        }

    def _create_synthetic_observations(self, **kwargs):
        statevec_true = kwargs["statevec_true"]
        nd, nt = statevec_true.shape

        # schedule
        obs_t, ind_m, m_obs = fake_generate_observation_schedule(**kwargs)
        ind_m = np.asarray(ind_m, dtype=int)
        print("OBS columns:", ind_m)

        # indexing
        _, indx_map, _ = fake_icesee_get_index(**kwargs)
        vec_inputs = kwargs["vec_inputs"]

        hdim = kwargs["hdim"]

        # outputs
        hu_obs = np.zeros((nd, m_obs))
        error_R = np.zeros((nd, m_obs * 2 + 1))

        # fill error_R
        for i, sig in enumerate(self.params["sig_obs"]):
            start = i * hdim
            end = start + hdim
            error_R[start:end, :] = sig

        # identify vars
        bed_alias = {"bed"}
        key_is_bed = {k: (k in bed_alias) for k in vec_inputs}

        # mask (only 1 point)
        bed_mask_map = {"bed": np.zeros(hdim, dtype=bool)}
        bed_mask_map["bed"][hdim // 2] = True

        # time → obs column
        bed_time_to_col = {t: c for c, t in enumerate(ind_m)}

        obs_set = set(kwargs["observed_vars"] + kwargs["observed_params"])

        km = 0
        for step in range(nt):
            if km < m_obs and (step + 1 == ind_m[km]):
                for key in vec_inputs:
                    idx = indx_map[key]
                    bed_flag = key_is_bed[key]

                    if key in obs_set and not bed_flag:
                        sigma = error_R[idx, km]
                        hu_obs[idx, km] = statevec_true[idx, step+1] + np.random.normal(0, sigma, size=idx.size)

                    elif bed_flag and (step+1 in bed_time_to_col):
                        col = bed_time_to_col[step+1]
                        mask = bed_mask_map["bed"]
                        idx_obs = idx[mask]
                        sigma_obs = error_R[idx_obs, col]
                        hu_obs[idx_obs, col] = statevec_true[idx_obs, step+1] + np.random.normal(0, sigma_obs)

                km += 1

        return hu_obs, error_R.T


# -------------------------------------------------------------------
# Helper for debugging
# -------------------------------------------------------------------
def indx_map_example(hdim):
    keys = ["h","u","v","bed","smb"]
    idx={}
    for i,k in enumerate(keys):
        idx[k] = np.arange(i*hdim,(i+1)*hdim)
    return idx


# -------------------------------------------------------------------
# RUN TEST
# -------------------------------------------------------------------
def run_test():

    model = TestICESEE()

    hdim = 100
    vec_inputs = ["h", "u", "v", "bed", "smb"]
    nd = hdim * len(vec_inputs)
    nt = 20

    statevec_true = np.zeros((nd, nt))
    for i in range(nd):
        statevec_true[i, :] = np.linspace(0, 1, nt)

    hu_obs, R = model._create_synthetic_observations(
        statevec_true=statevec_true,
        params=model.params,
        nd=nd,
        nt=nt,
        hdim=hdim,
        vec_inputs=vec_inputs,
        observed_vars=["h", "u", "v"],
        observed_params=["smb"],
        bed_obs_snapshot=[10],
        obs_times=[5, 15, 18],
    )

    bed_idx_mid = indx_map_example(hdim)["bed"][hdim//2]

    print("\nBED OBSERVATION RESULTS")
    print("----------------------------")
    print("bed index middle:", bed_idx_mid)
    print("hu_obs bed row:", hu_obs[bed_idx_mid, :])
    print("Non-zero obs columns:", np.nonzero(hu_obs[bed_idx_mid, :])[0])


# run the test
run_test()


OBS columns: [ 5 15 18]

BED OBSERVATION RESULTS
----------------------------
bed index middle: 350
hu_obs bed row: [0.26315789 0.78947368 0.94736842]
Non-zero obs columns: [0 1 2]


In [5]:
import numpy as np

# ============================================================
# Dummy icesee_get_index for testing
# ============================================================
def fake_icesee_get_index(**kwargs):
    nd = kwargs["nd"]
    vec_inputs = kwargs["vec_inputs"]
    nvars = len(vec_inputs)
    hdim = nd // nvars

    indx_map = {}
    for i, key in enumerate(vec_inputs):
        start = i * hdim
        end = start + hdim
        indx_map[key] = np.arange(start, end)

    return None, indx_map, None


# ============================================================
# Test class with correct H_matrix implementation
# ============================================================
class TestICESEE_H:
    def __init__(self):
        self.params = {}
        self.model_kwargs = {}

    def H_matrix(self, n_model):
        params = self.params
        vec_inputs = params["vec_inputs"]
        observed_vars = params["observed_vars"]
        observed_params = params["observed_params"]
        observed_set = set(observed_vars + observed_params)

        # full indexing
        _, indx_map, _ = fake_icesee_get_index(**self.model_kwargs)

        # observation schedule
        obs_t, ind_m, m_obs = self.model_kwargs["obs_schedule"]
        ind_m = np.asarray(ind_m, dtype=int)

        # bed settings
        bed_snaps = self.model_kwargs.get("bed_obs_snapshot", [])
        bed_mask_map = self.model_kwargs.get("bed_obs_mask_map", {})

        obs_list = []   # (obs_row, state_idx)

        for col, t in enumerate(ind_m):

            for key in vec_inputs:
                idx_vec = indx_map[key]

                # --- BED special case ---
                if key == "bed":
                    if t in bed_snaps:
                        mask = bed_mask_map[key]
                        idx_obs = idx_vec[mask]
                        for j in idx_obs:
                            obs_list.append((col, j))
                    continue

                # --- STANDARD observed variables ---
                if key in observed_set:
                    for j in idx_vec:
                        obs_list.append((col, j))

        # build H
        m_total = len(obs_list)
        H = np.zeros((m_total, n_model))

        for row_idx, (_, state_j) in enumerate(obs_list):
            H[row_idx, state_j] = 1.0

        return H


# ============================================================
# RUN TEST
# ============================================================
def run_test():
    hdim = 10
    vec_inputs = ["h", "u", "v", "bed", "smb"]
    nd = hdim * len(vec_inputs)

    observed_vars = ["h", "u", "v"]
    observed_params = ["smb"]

    # bed observed only at mid index
    bed_mask = np.zeros(hdim, dtype=bool)
    bed_mask[hdim // 2] = True   # only 1 point observed!

    # bed observed only at time step 10
    bed_snap_times = [10]

    # full observation schedule
    obs_times = [5, 10, 15]  # 3 obs cycles

    model = TestICESEE_H()

    model.params = {
        "vec_inputs": vec_inputs,
        "observed_vars": observed_vars,
        "observed_params": observed_params,
    }

    model.model_kwargs = {
        "vec_inputs": vec_inputs,
        "nd": nd,
        "obs_schedule": (obs_times, obs_times, len(obs_times)),
        "bed_obs_snapshot": bed_snap_times,
        "bed_obs_mask_map": {"bed": bed_mask},
    }

    # Build H
    H = model.H_matrix(nd)

    # ---------- Diagnostics ----------
    print("\n=== TEST RESULTS ===")
    print("H shape:", H.shape)

    # Total expected rows:
    # each obs time:
    #  h (10) + u (10) + v (10) + smb (10) = 40 rows
    # bed only appears at time 10, 1 masked index -> 1 row
    expected_rows = 40 * 3 + 1
    print("Expected rows:", expected_rows)

    # count bed rows
    bed_rows = np.where(H[:, vec_inputs.index("bed")*hdim + hdim//2] == 1)[0]
    print("bed observation row indices:", bed_rows)

    # Check that bed appears ONLY ONCE
    print("Total bed obs rows:", len(bed_rows))

    # Check no other bed indices appear
    bed_block = np.arange(3*hdim, 4*hdim)
    other_bed_indices = np.setdiff1d(bed_block, [bed_block[hdim//2]])

    for idx in other_bed_indices:
        if np.any(H[:, idx] == 1):
            print("ERROR: Unobserved bed index appeared:", idx)
            break
    else:
        print("✔ Unobserved bed indices = correctly absent.")

    print("\n>>> Basic test complete.")


# Run test
run_test()



=== TEST RESULTS ===
H shape: (121, 50)
Expected rows: 121
bed observation row indices: [70]
Total bed obs rows: 1
✔ Unobserved bed indices = correctly absent.

>>> Basic test complete.


In [1]:
import numpy as np

# -----------------------------
# 1. toy state vector structure
# -----------------------------
hdim = 4
vec_inputs = ['h','s','u','v','bed','fric','smb']

# Build fake index map for each variable
indx_map = {}
start = 0
for key in vec_inputs:
    indx_map[key] = np.arange(start, start + hdim)
    start += hdim

# Show index structure
print("\nIndex Map:")
for k, v in indx_map.items():
    print(f"{k:5s} -> {v}")

# -----------------------------
# 2. Observed variables
# -----------------------------
observed = ['h','u','bed']

# -----------------------------
# 3. Toy BED MASK
# -----------------------------
# observe only points 0 and 2
bed_mask_map = {
    'bed': np.array([True, False, True, False], dtype=bool)
}
print("\nBed mask:", bed_mask_map['bed'])

# -----------------------------
# 4. Build H (toy version)
# -----------------------------
all_obs_indices = []
for key in observed:

    idx = indx_map[key]

    if key == "bed":
        # apply mask
        mask = bed_mask_map['bed']
        idx = idx[mask]

    all_obs_indices.append(idx)

obs_indices = np.concatenate(all_obs_indices)
print("\nObservation indices:", obs_indices)

# -----------------------------
# 5. Build the H matrix
# -----------------------------
n_model = len(vec_inputs) * hdim
m_obs = obs_indices.size

H = np.zeros((m_obs, n_model))
H[np.arange(m_obs), obs_indices] = 1.0

print("\nH shape:", H.shape)
print("\nH matrix:\n", H)

# -----------------------------
# 6. VALIDATION CHECKS
# -----------------------------
# test 1: H has exactly one "1" per row
row_sums = H.sum(axis=1)
assert np.allclose(row_sums, 1.0), "Row sums must be 1 (each row is one observation)"

# test 2: bed rows correspond only to indices selected by mask
expected_bed_indices = indx_map['bed'][bed_mask_map['bed']]
actual_bed_indices = obs_indices[-expected_bed_indices.size:]

print("\nExpected bed indices:", expected_bed_indices)
print("Actual bed indices from H:", actual_bed_indices)

assert np.all(expected_bed_indices == actual_bed_indices), \
       "Mismatch: H bed indices do not match mask!"

print("\n✔️ All tests passed!")



Index Map:
h     -> [0 1 2 3]
s     -> [4 5 6 7]
u     -> [ 8  9 10 11]
v     -> [12 13 14 15]
bed   -> [16 17 18 19]
fric  -> [20 21 22 23]
smb   -> [24 25 26 27]

Bed mask: [ True False  True False]

Observation indices: [ 0  1  2  3  8  9 10 11 16 18]

H shape: (10, 28)

H matrix:
 [[1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0.]
 

In [4]:
import numpy as np

# -------------------------------------------------------------------
# Fake icesee_get_index to emulate real behavior
# -------------------------------------------------------------------
def icesee_get_index(**kwargs):
    """
    Returns:
        vecs:   not used here
        indx_map: dict mapping each field to global indices in the statevector
        extra:  not used
    """
    return None, kwargs["indx_map"], None


# -------------------------------------------------------------------
# Dummy class holding params + the H_matrix function
# -------------------------------------------------------------------
class DummyICESEE:
    def __init__(self, params, model_kwargs):
        self.params = params
        self.model_kwargs = model_kwargs

    # Paste your H_matrix here
    def H_matrix(self, n_model):
        import numpy as np

        params = self.params
        observed = params["all_observed"]
        vec_inputs = params["vec_inputs"]

        vecs, indx_map, _ = icesee_get_index(**self.model_kwargs)
        bed_mask_map = self.model_kwargs.get("bed_mask_map", {})

        bed_aliases = {'bed', 'bedrock', 'bed_topography', 'bedtopo', 'bedtopography'}
        key_is_bed = {k: (k in bed_aliases) for k in vec_inputs}
        key_idx_map = {k: np.asarray(indx_map[k], dtype=int) for k in vec_inputs}

        all_obs_indices = []

        for key in observed:
            idx = np.asarray(indx_map[key], dtype=int)

            if key == "bed" and key in bed_mask_map:
                mask = np.asarray(bed_mask_map[key], dtype=bool)
                if mask.size != idx.size:
                    raise ValueError(
                        f"bed_mask_map['{key}'] length {mask.size} does not match bed vector length {idx.size}"
                    )
                idx = idx[mask]

            all_obs_indices.append(idx)

        obs_indices = np.concatenate(all_obs_indices).astype(int)

        if obs_indices.max() >= n_model:
            raise ValueError(f"H_matrix error: obs index {obs_indices.max()} >= state size {n_model}")

        m_obs = obs_indices.size
        H = np.zeros((m_obs, n_model))
        H[np.arange(m_obs), obs_indices] = 1.0
        return H


# -------------------------------------------------------------------
# TEST 1: bed mask keeps only selected indices
# -------------------------------------------------------------------
def test_H_matrix_with_bed_mask():
    # Global statevector of length 10:
    # h:   0,1
    # u:   2,3
    # v:   4,5
    # bed: 6,7,8,9
    indx_map = {
        "h":   np.array([0,1]),
        "u":   np.array([2,3]),
        "v":   np.array([4,5]),
        "bed": np.array([6,7,8,9])
    }

    params = {
        "all_observed": ["h", "u", "bed"],     # observe h, u, and only masked bed points
        "vec_inputs":   ["h", "u", "v", "bed"]
    }

    # Mask bed to keep only idx positions 0 and 2 → indices 6 and 8
    bed_mask_map = {
        "bed": np.array([1, 0, 1, 0], dtype=bool)
    }

    model_kwargs = {
        "indx_map": indx_map,
        "bed_mask_map": bed_mask_map
    }

    model = DummyICESEE(params, model_kwargs)
    H = model.H_matrix(n_model=10)

    # --- Expected obs_indices = [0,1] + [2,3] + [6,8] ---
    expected = np.array([0,1, 2,3, 6,8])

    # --- Tests ---
    assert H.shape == (expected.size, 10)
    recovered = np.where(H == 1.0)[1]  # column indices of the 1s
    assert np.array_equal(recovered, expected)

    print("TEST 1 PASSED: bed mask applied correctly and H is structurally valid.")


# -------------------------------------------------------------------
# TEST 2: bed mask excludes ALL bed points
# -------------------------------------------------------------------
def test_H_matrix_bed_fully_masked():
    indx_map = {
        "h":   np.array([0,1]),
        "u":   np.array([2,3]),
        "bed": np.array([4,5,6])
    }

    params = {
        "all_observed": ["h", "u", "bed"],
        "vec_inputs":   ["h","u","bed"]
    }

    # Mask removes all bed entries
    bed_mask_map = {
        "bed": np.array([0,0,0], dtype=bool)
    }

    model_kwargs = {
        "indx_map": indx_map,
        "bed_mask_map": bed_mask_map
    }

    model = DummyICESEE(params, model_kwargs)
    H = model.H_matrix(n_model=7)

    expected = np.array([0,1,2,3])  # Only h and u remain

    recovered = np.where(H == 1.0)[1]
    assert np.array_equal(recovered, expected)

    print("TEST 2 PASSED: fully masked bed removed correctly.")


# -------------------------------------------------------------------
# Run tests manually
# -------------------------------------------------------------------
if __name__ == "__main__":
    test_H_matrix_with_bed_mask()
    test_H_matrix_bed_fully_masked()
    print("\nALL TESTS PASSED.")


TEST 1 PASSED: bed mask applied correctly and H is structurally valid.
TEST 2 PASSED: fully masked bed removed correctly.

ALL TESTS PASSED.
